# IMDB sentiment: TF-IDF baseline vs fine-tuned DistilBERT

Runs end to end on Colab's free GPU: **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all** (~10 min with the default 5k subset).
Stages: 1 Setup, 2 Data, 3 Baseline, 4 Transformer, 5 Evaluation, 6 Error analysis, 7 Inference + demo, 8 Save, Hub, README.

## 1. Setup

**Why:** a project is only credible if someone else can rerun it and get the same numbers. So we install in one cell, fail fast if there is no GPU (training on CPU takes hours), and fix every random seed before anything random happens.

**Watch out**
- Run the install cell *before* importing `transformers`. If Colab asks for a restart after it, restart once and skip the install cell.
- `set_seed` makes results repeatable, not bit-identical: GPU kernels are slightly non-deterministic, so expect about ±0.3 accuracy points between reruns.
- Colab's disk is wiped when the runtime disconnects. Stage 8 pushes the model to the Hub so you never lose it.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate gradio tabulate

In [ ]:
import re, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import set_seed

SEED = 42
set_seed(SEED)  # seeds python, numpy and torch (CPU + GPU) in one call

assert torch.cuda.is_available(), "No GPU found: Runtime > Change runtime type > T4 GPU, then rerun."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

ROOT = Path("/content/imdb-sentiment")
MODEL_DIR, OUT = ROOT / "model", ROOT / "outputs"   # model = final weights, outputs = plots + README assets
for d in (MODEL_DIR, OUT):
    d.mkdir(parents=True, exist_ok=True)
print([str(p) for p in ROOT.iterdir()])

## 2. Data

**Why:** look at the data before modelling. Class balance decides whether accuracy is a safe headline metric, and the length distribution decides how aggressively we must truncate for the transformer in stage 4.

**The split (three disjoint sets):** 5,000 train, 500 validation, 1,000 test. Validation is carved from the *training* pool, so the test set is touched exactly once, at the end.

**Scaling up later:** change only `N_TRAIN` / `N_TEST` below. `N_TRAIN` can go up to 24,500 (25,000 minus the validation reviews) and `N_TEST` up to 25,000. Full data takes roughly 5x longer to train (~15-25 min on a T4). Nothing else needs to change.

**Watch out**
- The raw IMDB files are **sorted by label**. Taking the first 1,000 rows without shuffling gives you a test set of only negative reviews. We shuffle first.
- Reviews contain literal `<br />` HTML tags; we replace them with spaces.
- `datasets` can return lazy columns, so we convert to plain lists/arrays once and use those everywhere.

In [ ]:
from datasets import load_dataset

N_TRAIN, N_VAL, N_TEST = 5000, 500, 1000
LABELS = ["negative", "positive"]   # IMDB: 0 = negative, 1 = positive

raw = load_dataset("stanfordnlp/imdb")
pool = raw["train"].shuffle(seed=SEED)
assert N_TRAIN + N_VAL <= len(pool), "N_TRAIN + N_VAL must fit inside the 25,000-review training pool"

clean = lambda ex: {"text": ex["text"].replace("<br />", " ")}
train = pool.select(range(N_TRAIN)).map(clean)
val = pool.select(range(N_TRAIN, N_TRAIN + N_VAL)).map(clean)              # disjoint from train
test = raw["test"].shuffle(seed=SEED).select(range(N_TEST)).map(clean)     # separate split entirely

X_train, y_train = list(train["text"]), np.array(train["label"])
X_val, y_val = list(val["text"]), np.array(val["label"])
X_test, y_test = list(test["text"]), np.array(test["label"])
print(len(X_train), len(X_val), len(X_test))

In [ ]:
counts = pd.DataFrame({n: Counter(y) for n, y in [("train", y_train), ("val", y_val), ("test", y_test)]})
counts = counts.sort_index(); counts.index = [LABELS[i] for i in counts.index]
print(counts, "\n")

words = pd.Series([len(t.split()) for t in X_train])
print(words.describe().round(0).to_string())
print(f"\n{(words > 190).mean():.0%} of reviews are longer than ~190 words (about 256 tokens)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
counts.T.plot.bar(ax=ax[0], rot=0, title="Class balance")
ax[1].hist(words.clip(upper=1000), bins=50)
ax[1].axvline(190, color="red", ls="--", label="~256 tokens")
ax[1].set(title="Review length (words)", xlabel="words", ylabel="reviews"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "data_overview.png", dpi=120); plt.show()

## 3. Baseline: TF-IDF + Logistic Regression

**Why:** a baseline tells you what the fancy model has to beat. If a 10-second linear model gets 88%, a transformer at 91% is a real but modest win, and you can only say that if the baseline is honest (tuned on validation, not on test).

**Choices**
| Setting | Value | Why |
|---|---|---|
| `ngram_range` | (1, 2) | Unigrams plus bigrams capture short phrases like "not good" that single words miss. |
| `min_df` | 2 | Drops words seen in only one review: mostly typos and names, pure noise. |
| `max_features` | 100,000 | Caps memory and runtime; bigrams otherwise explode the vocabulary. |
| `sublinear_tf` | True | Uses 1 + log(count), so a word repeated 10 times is not counted 10x as important. |
| `C` | best of 1, 10, 100 on **validation** | Inverse regularisation strength. TF-IDF values are small, so larger `C` often helps; we let validation pick instead of guessing. |
| `max_iter` | 1000 | Default 100 often stops before converging on 100k features. |

**Watch out (data leakage):** the vectorizer must be fitted on training text only. `make_pipeline` guarantees this because `.fit` only ever sees the training data and `.predict` just applies it. Fitting TF-IDF on train+test (a very common mistake) leaks test vocabulary and IDF statistics into the model.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             precision_recall_fscore_support, ConfusionMatrixDisplay)

def make_base(C):
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True),
        LogisticRegression(C=C, max_iter=1000),
    )

val_acc = {C: make_base(C).fit(X_train, y_train).score(X_val, y_val) for C in (1, 10, 100)}
best_C = max(val_acc, key=val_acc.get)
print("validation accuracy by C:", {C: round(a, 4) for C, a in val_acc.items()}, "-> using C =", best_C)

t0 = time.time(); base = make_base(best_C).fit(X_train, y_train); base_train_s = time.time() - t0
t0 = time.time(); base_pred = base.predict(X_test); base_infer_s = time.time() - t0

print(f"\nTest accuracy: {accuracy_score(y_test, base_pred):.4f}  (train {base_train_s:.1f}s)\n")
print(classification_report(y_test, base_pred, target_names=LABELS, digits=4))

def summarize(name, y, pred, train_s, infer_s):
    """One table row: overall + per-class metrics + cost. Reused for both models in stage 5."""
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1])
    row = {"Model": name, "Accuracy": accuracy_score(y, pred), "Macro-F1": f.mean()}
    for i, l in enumerate(LABELS):
        row |= {f"{l} precision": p[i], f"{l} recall": r[i], f"{l} F1": f[i]}
    return row | {"Train time (s)": train_s, "Test inference (s)": infer_s}

## 4. Transformer: fine-tune DistilBERT

**Why:** DistilBERT is BERT with 40% fewer parameters that keeps ~97% of its language understanding, so it trains in minutes on a free T4. Pretraining already taught it grammar and word meaning in context (including negation); fine-tuning only has to teach it the *task*.

**Tokenization, truncation, padding**
- The model turns text into subword token ids and can read at most 512 of them. **Truncation** cuts anything longer than `MAX_LEN`.
- A batch must be a rectangle, so shorter reviews get **padding** tokens, and an `attention_mask` tells the model to ignore them.
- We do **not** pad during `.map`. `DataCollatorWithPadding` pads each batch only to its own longest review (dynamic padding), which is much faster than padding everything to 256.
- `MAX_LEN = 256` is a speed/accuracy trade-off for the free GPU. 512 would be about 2-3x slower and typically adds around a point of accuracy. The cell below prints how many reviews 256 truncates, and stage 6 measures what that costs.

**Hyperparameters**
| Setting | Value | Why |
|---|---|---|
| `MAX_LEN` | 256 | See above. |
| `BATCH` (train) | 16 | Fits a T4 at length 256 with fp16. Bigger batches mean fewer weight updates on a small dataset. |
| eval batch | 64 | No gradients stored at eval, so memory allows 4x larger. Affects speed only. |
| `LR` | 2e-5 | Small because the weights are already good; large steps would erase pretraining. The BERT paper's range is 2e-5 to 5e-5. |
| `EPOCHS` | 3 | Within the paper's 2-4. More epochs on 5k reviews start memorising: watch validation loss rise while train loss keeps falling. |
| `weight_decay` | 0.01 | Mild AdamW regularisation, the standard BERT default. |
| `warmup_steps` | 50 | About 5% of 939 total steps. Starts the learning rate near zero so the freshly initialised classifier head does not wreck the pretrained layers with huge early gradients. |
| `fp16` | True | Roughly 2x faster and half the activation memory on a T4 at no meaningful accuracy cost. |
| `eval_steps` / `logging_steps` | 50 | Gives ~19 points per curve instead of 3. Validation uses the 500 held-out reviews, never test. |
| `seed` | 42 | Same seed as stage 1, so the classifier head initialises identically on rerun. |

**Watch out**
- The warning "Some weights were not initialized ... classifier" is **expected**: the 2-class head is new and is what we train.
- Use the *same* tokenizer as the model (`distilbert-base-uncased` lowercases text). A mismatched tokenizer silently produces garbage.
- Never pass the test set as `eval_dataset`. Doing so tunes on test.

In [ ]:
import evaluate
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODEL, MAX_LEN, BATCH, LR, EPOCHS = "distilbert-base-uncased", 256, 16, 2e-5, 3

tok = AutoTokenizer.from_pretrained(MODEL)
tokenize = lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN)   # no padding here: the collator does it per batch
train_tok = train.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test.map(tokenize, batched=True, remove_columns=["text"])

full_len = np.array([len(x) for x in tok(X_train)["input_ids"]])   # untruncated token counts (a length warning here is harmless)
print(f"{(full_len > MAX_LEN).mean():.0%} of training reviews exceed {MAX_LEN} tokens and will be truncated")
print(tok.decode(train_tok[0]["input_ids"][:25]), "...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2,
    id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)},  # so the Hub widget shows names, not LABEL_0
)

metric = evaluate.combine(["accuracy", "f1"])
compute_metrics = lambda p: metric.compute(predictions=p.predictions.argmax(-1), references=p.label_ids)

args = TrainingArguments(
    output_dir=str(OUT / "trainer"), num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64, learning_rate=LR, weight_decay=0.01, warmup_steps=50,
    fp16=True, eval_strategy="steps", eval_steps=50, logging_steps=50,
    save_strategy="no", report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics, processing_class=tok)

t0 = time.time(); trainer.train(); bert_train_s = time.time() - t0
print(f"\nTraining took {bert_train_s / 60:.1f} min")

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
tr, ev = log.dropna(subset=["loss"]), log.dropna(subset=["eval_loss"])
print(ev[["step", "eval_loss", "eval_accuracy", "eval_f1"]].round(4).tail(6).to_string(index=False))

plt.figure(figsize=(6, 3.5))
plt.plot(tr["step"], tr["loss"], label="train loss"); plt.plot(ev["step"], ev["eval_loss"], label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("DistilBERT fine-tuning"); plt.legend()
plt.tight_layout(); plt.savefig(OUT / "loss_curves.png", dpi=120); plt.show()

**Reading the curve:** both lines should fall quickly then flatten. If validation loss turns up while train loss keeps dropping, the model is overfitting; lower `EPOCHS` or add data. A tiny gap between the two means you could train longer or on more data.

## 5. Evaluation: both models, one table

**Why:** a single accuracy number hides which class a model gets wrong and what it costs. We compute identical metrics for both models on the **same** untouched test set, and include training and inference time, because "is it worth it?" is the actual engineering question.

**Watch out**
- `trainer.predict` runs the model in eval mode (dropout off) for you. In your *own* inference code you must call `model.eval()` yourself (stage 7), or dropout stays on and predictions become random.
- With 1,000 test reviews, accuracy has a 95% margin of error of about ±2 points. Do not over-read a 0.5-point difference.

In [ ]:
t0 = time.time(); out = trainer.predict(test_tok); bert_infer_s = time.time() - t0
logits = out.predictions
bert_pred = logits.argmax(-1)

results = pd.DataFrame([
    summarize("TF-IDF + LogReg", y_test, base_pred, base_train_s, base_infer_s),
    summarize("DistilBERT (fine-tuned)", y_test, bert_pred, bert_train_s, bert_infer_s),
])
print(results.set_index("Model").round(3).T.to_string())

a, b = results.iloc[0], results.iloc[1]
moe = 1.96 * np.sqrt(b["Accuracy"] * (1 - b["Accuracy"]) / N_TEST) * 100
print(f"\nDistilBERT: {(b['Accuracy'] - a['Accuracy']) * 100:+.1f} accuracy points for "
      f"{b['Train time (s)'] / a['Train time (s)']:.0f}x the training time and "
      f"{b['Test inference (s)'] / a['Test inference (s)']:.0f}x the inference time (95% margin of error ~±{moe:.1f} pts)")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for axis, name, pred in zip(ax, results["Model"], (base_pred, bert_pred)):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=LABELS, ax=axis, colorbar=False)
    axis.set_title(name)
plt.tight_layout(); plt.savefig(OUT / "confusion_matrices.png", dpi=120); plt.show()

## 6. Error analysis (the most important stage)

**Why:** metrics say *how often* the model fails; error analysis says *why*, and that is what separates a portfolio project from a tutorial. Reading real failures tells you whether to fix the data, the truncation, or the model.

**Method**
1. Take all of DistilBERT's test mistakes and sort by confidence. The most **confident** mistakes are the most informative: the model is not unsure, it is systematically wrong.
2. A small rule set gives every error a first-pass pattern tag (negation, mixed sentiment, sarcasm, truncated). Rules are crude, so **read the 20 printed reviews yourself** and correct tags through `MANUAL` (example: `MANUAL = {412: ["sarcasm"]}`, keyed by the `#index` printed beside each review), then rerun the cell.
3. The cell compares each pattern's **error rate when present** with the overall error rate. Words like "but" appear in most reviews, so a pattern only counts as an explanation if reviews carrying it fail more often than average. The README states this too.
4. It also checks where the two models disagree.

**Things to look for**
- *Negation:* "not bad", "can't say I hated it", polarity flips on words the model strongly associates with one class.
- *Mixed sentiment:* praise for acting, scorn for the plot. The label is the author's overall verdict, often stated in one sentence near the end.
- *Sarcasm / irony:* positive words, negative meaning.
- *Truncation:* verdicts at the end of long reviews that the model never saw.
- *Label noise and plot summaries:* some "wrong" predictions are correct, and dark plot descriptions inside positive reviews fool the model.

In [ ]:
probs = torch.softmax(torch.tensor(logits), -1).numpy()
conf = probs.max(1)
wrong = np.flatnonzero(bert_pred != y_test)
tok_len = [len(x) for x in tok(X_test)["input_ids"]]   # untruncated lengths

CONTRAST = r"\b(?:but|however|although|though|despite|yet|unfortunately)\b"
POS = r"\b(?:good|great|excellent|best|love|loved|brilliant|beautiful|enjoyed|funny)\b"
NEG = r"\b(?:bad|worst|boring|terrible|awful|waste|poor|disappointing|disappointed|dull)\b"
RULES = {
    "negation": lambda t, n: bool(re.search(r"(?:\bnot|\bnever|\bno|n't)\s+(?:\w+\s+){0,2}?(?:good|bad|great|terrible|awful|worst|best|boring|funny|worth|waste|recommend|original)\b", t)),
    "mixed sentiment": lambda t, n: all(re.search(p, t) for p in (CONTRAST, POS, NEG)),
    "sarcasm / irony": lambda t, n: bool(re.search(r"yeah,? right|oh,? (?:great|boy|joy)|so bad (?:it'?s|that it'?s) good|guilty pleasure|thanks a lot|what a (?:masterpiece|joke)", t)),
    "truncated (>%d tokens)" % MAX_LEN: lambda t, n: n > MAX_LEN,
}
MANUAL = {}   # after reading: {test_index: ["sarcasm / irony"]} overrides the auto tags for that review

def tags(i):
    return MANUAL.get(i) or [k for k, f in RULES.items() if f(X_test[i].lower(), tok_len[i])] or ["other"]

print(f"{len(wrong)} of {N_TEST} test reviews misclassified ({len(wrong) / N_TEST:.1%})")
print(f"false positives (neg -> pos): {((bert_pred == 1) & (y_test == 0)).sum()} | false negatives (pos -> neg): {((bert_pred == 0) & (y_test == 1)).sum()}\n")

top = wrong[np.argsort(-conf[wrong])][:20]
for i in top:
    print(f"#{i} true={LABELS[y_test[i]]} pred={LABELS[bert_pred[i]]} conf={conf[i]:.2f} tags={tags(i)}\n    {X_test[i][:500]}\n")

In [ ]:
b_wrong, t_wrong = base_pred != y_test, bert_pred != y_test
overall = t_wrong.mean()

pat_counts = Counter(k for i in wrong for k in tags(i))
auto = {k: np.array([f(X_test[i].lower(), tok_len[i]) for i in range(N_TEST)]) for k, f in RULES.items()}
rate = {k: (int((m & t_wrong).sum()), int(m.sum())) for k, m in auto.items()}   # (errors, reviews carrying the pattern)
pattern_df = pd.DataFrame(
    [(k, n, f"{n / len(wrong):.0%}", rate[k][1] if k in rate else "n/a",
      f"{rate[k][0] / max(rate[k][1], 1):.1%}" if k in rate else "n/a") for k, n in pat_counts.most_common()],
    columns=["pattern", "errors", "share of errors", "reviews with pattern", "error rate when present"])
print(f"Overall error rate: {overall:.1%}. A pattern only explains errors if its own error rate is clearly higher than that.")
print(pattern_df.to_string(index=False))

trunc = np.array(tok_len) > MAX_LEN
print(f"\nerror rate on truncated reviews:     {np.mean(bert_pred[trunc] != y_test[trunc]):.1%} (n={trunc.sum()})")
print(f"error rate on non-truncated reviews: {np.mean(bert_pred[~trunc] != y_test[~trunc]):.1%} (n={(~trunc).sum()})")
print(f"\nboth wrong: {(b_wrong & t_wrong).sum()} | only baseline wrong: {(b_wrong & ~t_wrong).sum()} | only DistilBERT wrong: {(~b_wrong & t_wrong).sum()}")

WHY = {
    "negation": "A negation flips the meaning of a word the model strongly associates with the opposite class. With only a few thousand fine-tuning examples it has seen few such constructions, so surface vocabulary can outweigh the negation.",
    "mixed sentiment": "These reviews praise some aspects and criticise others. The label is the author's overall verdict, which often rests on a single sentence. The model effectively averages the cues, so a strong run of praise or scorn can outvote it.",
    "sarcasm / irony": "Sarcasm uses positive words to express a negative opinion. Spotting it needs tone and world knowledge beyond local word context, and a 5k-review fine-tune barely teaches it.",
    "truncated (>%d tokens)" % MAX_LEN: "Reviews longer than the token limit are cut off. Reviewers often state the verdict at the end, which the model never sees, so it decides from the opening plot summary instead.",
    "other": "Not matched by any rule; read these manually. IMDB has some label noise, plot summaries full of negative words inside positive reviews, and comparisons to other films.",
}

### Write-up: how to turn the output into findings
For each pattern in the table (biggest first), write four things: **what** it is, **one quoted example**, **why** the model fails (the `WHY` dict is a starting point; edit it to match what *you* saw), and **what you would try** (e.g. head+tail truncation or `MAX_LEN=512` for truncation errors, more training data or a larger model for mixed sentiment, a sarcasm-specific dataset for irony). Stage 8 puts the counts, explanations and your three most confident mistakes straight into the README.

## 7. Inference and Gradio demo

**Why:** a model nobody can try is a notebook result, not a project. `predict` wraps tokenizer + model + softmax into one function, and Gradio puts a text box on top of it with a shareable link.

**Watch out**
- **`model.eval()`**: after `trainer.train()` the model is left in training mode, so dropout is still active and the same text can give different answers. We call `model.eval()` explicitly.
- **`torch.no_grad()`**: skips gradient bookkeeping, saving memory and time.
- Use the **same `MAX_LEN` truncation** as training. Confidence is a softmax probability, which is often overconfident; it is not a calibrated "chance of being right".

In [ ]:
LABEL_NAMES = LABELS

def scores(text):
    """Return {label: probability} for any review text."""
    if not text or not text.strip():
        raise ValueError("Please enter a review.")
    model.eval()
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
    with torch.no_grad():
        p = model(**enc).logits.softmax(-1)[0]
    return {LABEL_NAMES[i]: float(p[i]) for i in range(len(LABEL_NAMES))}

def predict(text):
    """Return (label, confidence)."""
    s = scores(text)
    label = max(s, key=s.get)
    return label, round(s[label], 4)

for t in ["An absolute masterpiece. The acting, the score, everything worked.",
          "Not bad at all, I expected to hate it.",
          "Oh great, another two hours of my life I will never get back."]:
    print(predict(t), "<-", t)

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=scores,
    inputs=gr.Textbox(lines=6, label="Movie review"),
    outputs=gr.Label(num_top_classes=2, label="Sentiment"),
    title="IMDB sentiment: fine-tuned DistilBERT",
    examples=[["A moving, beautifully acted film that stayed with me for days."],
              ["Two hours of my life I will never get back. The plot made no sense."],
              ["The acting was great, but the story was dull and the ending was a mess."]],
)
demo.launch(share=True)   # share=True gives a temporary public URL (valid ~72h) that works from Colab

## 8. Save, push to the Hugging Face Hub, README

**Why:** the Hub gives your model a permanent public URL, a live inference widget, and a model card, which is what a recruiter can actually click.

**One-time Hub setup**
1. Create a free account at huggingface.co.
2. Go to **Settings > Access Tokens > Create new token**, choose type **Write**, and copy it.
3. In Colab, click the **key icon** (Secrets) in the left sidebar > **Add new secret**: name `HF_TOKEN`, value = your token, and switch **Notebook access** on.
4. Run the cell below. It logs in, builds the repo id from your username, and pushes model + tokenizer.

**Watch out:** never paste the token into a cell or commit it to GitHub. Secrets keep it out of the notebook file.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi

trainer.save_model(str(MODEL_DIR)); tok.save_pretrained(str(MODEL_DIR))   # local copy (lost when Colab disconnects)

login(token=userdata.get("HF_TOKEN"))
REPO = f"{whoami()['name']}/distilbert-imdb-sentiment"
model.push_to_hub(REPO); tok.push_to_hub(REPO)
print("Pushed: https://huggingface.co/" + REPO)

In [ ]:
# Sanity check: load it back from the Hub like a stranger would
from transformers import pipeline
print(pipeline("text-classification", model=REPO, device=0)("A wonderful film with a great cast."))

**README:** the next cell writes `README.md` from the *actual* variables in this run (results table, error patterns, your most confident mistakes), so the numbers cannot drift from the code. Read the generated file and edit the prose if your error analysis found something the template does not say. It also uploads a Hub model card with the same content.

In [ ]:
res_md = results.set_index("Model").round(3).T.to_markdown()
def finding(k, n):
    e, t = rate.get(k, (0, 0))
    r = e / t if t else None
    if r is not None and r <= overall:
        return (f"- **{k}** ({n} of {len(wrong)} errors): reviews with this pattern are not misclassified more often than average "
                f"({r:.0%} vs {overall:.0%}), so it does not explain the errors here.")
    extra = f" Error rate when present: {r:.0%} vs {overall:.0%} overall." if r is not None else ""
    return f"- **{k}** ({n} of {len(wrong)} errors): {WHY.get(k, WHY['other'])}{extra}"
findings = "\n".join(finding(k, n) for k, n in pat_counts.most_common())
examples = "\n\n".join(
    f"> **true: {LABELS[y_test[i]]}, predicted: {LABELS[bert_pred[i]]} ({conf[i]:.0%} confident)**: "
    f"{X_test[i][:300].replace(chr(10), ' ').strip()}..." for i in top[:3])
gain = (b["Accuracy"] - a["Accuracy"]) * 100

README = f"""# Sentiment analysis on IMDB reviews: TF-IDF baseline vs fine-tuned DistilBERT

## Problem
Classify movie reviews as positive or negative, and measure whether a fine-tuned transformer is worth its extra cost over a classic baseline. The main goal is understanding *where and why* the model fails, not just the headline accuracy.

## Approach
- **Data:** IMDB reviews ({N_TRAIN:,} train / {N_VAL} validation / {N_TEST:,} test), shuffled with seed {SEED}. Validation is carved from the training pool and the test set is used once. `<br />` tags are stripped.
- **Baseline:** TF-IDF (1-2 grams, sublinear tf) + Logistic Regression, `C={best_C}` chosen on validation.
- **Transformer:** `distilbert-base-uncased` fine-tuned with the Hugging Face `Trainer` for {EPOCHS} epochs (learning rate {LR}, batch size {BATCH}, max length {MAX_LEN}, fp16) on a free Colab T4.
- **Evaluation:** identical metrics on the same held-out test set, plus cost, plus manual error analysis.

## Results (test set, {N_TEST:,} reviews)
{res_md}

DistilBERT scores {gain:+.1f} accuracy points versus the baseline at {b['Train time (s)'] / a['Train time (s)']:.0f}x the training time. With {N_TEST:,} test reviews the 95% margin of error is about ±{moe:.1f} points, so differences smaller than that are noise.

![confusion matrices](outputs/confusion_matrices.png)
![loss curves](outputs/loss_curves.png)

## Error analysis
DistilBERT misclassified {len(wrong)} of {N_TEST:,} test reviews. Error patterns (a review can match several):

{pattern_df.to_markdown(index=False)}

Overall error rate is {overall:.1%}; the last column shows how often reviews carrying each pattern fail.

**What the model struggles with, and why**
{findings}

**Most confident mistakes**

{examples}

## How to run
1. Open `sentiment_analysis.ipynb` in Google Colab.
2. **Runtime > Change runtime type > T4 GPU**.
3. **Runtime > Run all** (about 10 minutes). To scale up, change `N_TRAIN` / `N_TEST` in the data cell.
4. To push to the Hub, add a Write token as a Colab secret named `HF_TOKEN`.

Model: https://huggingface.co/{REPO}

## Limitations and next steps
- Sentences beyond {MAX_LEN} tokens are truncated; try `MAX_LEN=512` or head+tail truncation.
- Trained on {N_TRAIN:,} reviews; the full 25,000 would likely help mixed-sentiment cases.
- Softmax confidence is not calibrated; add temperature scaling before using it as a probability.
- Movie reviews only: expect a drop on other domains (products, tweets).
"""
(ROOT / "README.md").write_text(README)

HUB_CARD = """---
language: en
license: apache-2.0
base_model: distilbert-base-uncased
datasets: [stanfordnlp/imdb]
metrics: [accuracy, f1]
pipeline_tag: text-classification
tags: [sentiment-analysis]
---
""" + README
HfApi().upload_file(path_or_fileobj=HUB_CARD.encode(), path_in_repo="README.md", repo_id=REPO)
print(README[:1500])

In [ ]:
# Download README + plots to commit alongside this notebook on GitHub
from google.colab import files
!cd /content/imdb-sentiment && zip -qr ../portfolio.zip README.md outputs/*.png
files.download("/content/portfolio.zip")